# Module 12: Feature Store to Model Registry

**Snowpark ML Fundamentals — Week 3 | Lunch & Learn**

---

## Learning Objectives
- Build a training set from the Feature Store
- Train a model on Feature Store features
- Register the model and attach an alias
- Run inference with Feature Store feature retrieval

## Key Concept
> The **Feature Store + Model Registry** integration creates a closed loop:
> features are computed once, used for training, stored in the registry,
> and the same features are retrieved at inference time — guaranteeing
> **training-serving consistency**.

---

In [1]:
%load_ext autoreload
%autoreload 2

## 12.1 Setup

Initialize both the Feature Store and Model Registry.
Re-create feature tables idempotently so this notebook works standalone.

In [2]:
import sys
sys.path.insert(0, '../src')

import logging
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.getLogger("snowflake.snowpark").setLevel(logging.ERROR)
logging.getLogger("snowflake.ml").setLevel(logging.ERROR)

from snowpark_fundamentals.session import create_session
from snowpark_fundamentals.feature_store.entities import (
    setup_feature_store,
    create_customer_entity,
    register_entity,
)
from snowpark_fundamentals.feature_store.feature_data import (
    create_customer_transactions_dataset,
    create_customer_interactions_dataset,
    create_rfm_features,
    create_behavioral_features,
)
from snowpark_fundamentals.feature_store.feature_views import (
    create_external_feature_view,
    register_feature_view,
    get_feature_view,
)
from snowpark_fundamentals.registry.model_registry import (
    get_registry,
    log_model,
    delete_model,
    set_model_alias,
    set_model_comment,
    get_model_by_alias,
)

session = create_session(env_path='../.env')

# Initialize Feature Store
fs = setup_feature_store(session)
print("Feature Store initialized.")

# Initialize Model Registry
current_db = session.get_current_database().replace('"', '')
current_schema = session.get_current_schema().replace('"', '')
registry = get_registry(session, current_db, current_schema)
print("Model Registry initialized.")

Feature Store initialized.
Model Registry initialized.


In [3]:
# Create source data (idempotent — uses CREATE OR REPLACE)
create_customer_transactions_dataset(session, n_rows=50000)
create_customer_interactions_dataset(session, n_rows=100000)

# Compute feature tables
rfm_df = create_rfm_features(session)
behavioral_df = create_behavioral_features(session)
print(f"RFM features: {rfm_df.count()} customers")
print(f"Behavioral features: {behavioral_df.count()} customers")

# Register entity and feature views (overwrite=True for idempotency)
customer_entity = create_customer_entity(desc="Customer entity for churn prediction")
customer_entity = register_entity(fs, customer_entity)

rfm_fv = create_external_feature_view(
    name="CUSTOMER_RFM_FV",
    entities=[customer_entity],
    feature_df=rfm_df,
    desc="RFM features: recency, frequency, monetary",
    timestamp_col="_FEATURE_TIMESTAMP",
)
rfm_fv = register_feature_view(fs, rfm_fv, version="V1", overwrite=True)

behavioral_fv = create_external_feature_view(
    name="CUSTOMER_BEHAVIORAL_FV",
    entities=[customer_entity],
    feature_df=behavioral_df,
    desc="Behavioral engagement features",
    timestamp_col="_FEATURE_TIMESTAMP",
)
behavioral_fv = register_feature_view(fs, behavioral_fv, version="V1", overwrite=True)

print("Feature views registered.")

RFM features: 2000 customers
Behavioral features: 2000 customers
Feature views registered.


## 12.2 Generate Training Set

Create a spine of customer IDs, then join features from both feature views.

In [4]:
from snowpark_fundamentals.feature_store.training_sets import (
    create_spine_dataframe,
    generate_training_set,
)

# Create spine from the RFM features table
spine_df = create_spine_dataframe(
    session,
    entity_table="CUSTOMER_RFM_FEATURES",
    entity_key="CUSTOMER_ID",
)

# Get registered feature views
rfm_fv = get_feature_view(fs, "CUSTOMER_RFM_FV", "V1")
behavioral_fv = get_feature_view(fs, "CUSTOMER_BEHAVIORAL_FV", "V1")

# Generate training set
training_set = generate_training_set(
    fs=fs,
    spine_df=spine_df,
    features=[rfm_fv, behavioral_fv],
    name="CHURN_FS_TRAINING_SET",
)

training_df = training_set.read.to_snowpark_dataframe()
print(f"Training set: {training_df.count()} rows, {len(training_df.columns)} columns")
print(f"Columns: {training_df.columns}")

Training set: 2000 rows, 23 columns
Columns: ['CUSTOMER_ID', 'DAYS_SINCE_LAST_ORDER', 'ORDERS_30D', 'ORDERS_90D', 'ORDERS_365D', 'ORDERS_TOTAL', 'SPEND_30D', 'SPEND_90D', 'SPEND_365D', 'SPEND_TOTAL', 'AVG_ORDER_VALUE', 'TOTAL_ITEMS', 'AVG_ITEMS_PER_ORDER', 'TOTAL_PAGE_VIEWS', 'TOTAL_CLICKS', 'TOTAL_SUPPORT_TICKETS', 'TOTAL_EMAIL_ENGAGEMENTS', 'INTERACTIONS_30D', 'SUPPORT_TICKETS_30D', 'INTERACTIONS_90D', 'DAYS_SINCE_LAST_INTERACTION', 'PREFERRED_CHANNEL', 'AVG_DURATION_SECONDS']


## 12.3 Train on Feature Store Data

Join the training features with churn labels, then train an XGBoost model.
We generate a probabilistic churn label based on recency, spending, and
engagement — with noise to avoid trivial separation.

In [5]:
from snowflake.snowpark import functions as F
from snowpark_fundamentals.data.loader import split_data
from snowpark_fundamentals.modeling.evaluation import evaluate_binary_classifier

# Create a churn label from Feature Store features
# Uses multiple risk signals + noise for realistic (non-trivial) separation
labeled_df = (
    training_df
    .with_column("_NOISE", F.uniform(0.0, 1.0, F.random(42)))
    .with_column(
        "CHURNED",
        F.when(
            # High-risk: long absence + no recent orders (85% churn rate)
            (F.col("DAYS_SINCE_LAST_ORDER") > 60) & (F.col("ORDERS_30D") == 0)
            & (F.col("_NOISE") < F.lit(0.85)),
            F.lit(1)
        ).when(
            # Medium-risk: moderate absence + low spend (25% churn rate)
            (F.col("DAYS_SINCE_LAST_ORDER") > 30)
            & (F.col("SPEND_30D") < F.lit(500))
            & (F.col("_NOISE") < F.lit(0.25)),
            F.lit(1)
        ).when(
            # Baseline noise: 5% of low-risk customers churn unexpectedly
            F.col("_NOISE") < F.lit(0.05),
            F.lit(1)
        ).otherwise(F.lit(0))
    )
    .drop("_NOISE")
)

# Select numeric features for training
FEATURE_COLS = [
    'DAYS_SINCE_LAST_ORDER', 'ORDERS_30D', 'ORDERS_90D', 'ORDERS_365D',
    'ORDERS_TOTAL', 'SPEND_30D', 'SPEND_90D', 'SPEND_365D', 'SPEND_TOTAL',
    'AVG_ORDER_VALUE', 'TOTAL_ITEMS', 'AVG_ITEMS_PER_ORDER',
    'TOTAL_PAGE_VIEWS', 'TOTAL_CLICKS', 'TOTAL_SUPPORT_TICKETS',
    'TOTAL_EMAIL_ENGAGEMENTS', 'INTERACTIONS_30D', 'SUPPORT_TICKETS_30D',
    'INTERACTIONS_90D', 'DAYS_SINCE_LAST_INTERACTION', 'AVG_DURATION_SECONDS',
]
LABEL_COL = 'CHURNED'

# Split
train_df, test_df = split_data(labeled_df.select(FEATURE_COLS + [LABEL_COL]))
print(f"Train: {train_df.count()}, Test: {test_df.count()}")

Train: 1569, Test: 431


In [6]:
from snowpark_fundamentals.modeling.trainer import train_model, predict

# Train XGBoost on Feature Store features
model = train_model(
    train_df, FEATURE_COLS, LABEL_COL,
    model_type='xgboost',
    model_params={'n_estimators': 200, 'max_depth': 8, 'learning_rate': 0.05},
)

preds = predict(model, test_df)
metrics = evaluate_binary_classifier(preds, LABEL_COL, 'PREDICTION')
print("Model metrics:", metrics)

Model metrics: {'accuracy': 0.860789, 'precision': 0.8558558558558559, 'recall': 0.6834532374100719, 'f1_score': 0.7600000000000001}


## 12.4 Register the Model

Register the Feature Store-trained model, set a production alias, and add a comment.

In [7]:
# Clean up from previous runs (idempotent)
try:
    delete_model(registry, 'CHURN_FS_MODEL')
    print("Deleted existing CHURN_FS_MODEL (re-run cleanup)")
except Exception:
    pass  # Model doesn't exist yet — first run

# Register
model_version = log_model(
    registry=registry,
    model=model.to_xgboost(),
    model_name='CHURN_FS_MODEL',
    version_name='V1',
    sample_input=test_df.select(FEATURE_COLS).limit(10),
    metrics=metrics,
)
print("Registered CHURN_FS_MODEL V1")

# Set production alias
set_model_alias(registry, 'CHURN_FS_MODEL', 'V1', 'production')

# Add context
set_model_comment(
    registry, 'CHURN_FS_MODEL',
    'Churn model trained on Feature Store features (RFM + behavioral). '
    'Uses feature retrieval for training-serving consistency.',
)
print("Alias 'production' set. Comment added.")

Deleted existing CHURN_FS_MODEL (re-run cleanup)
Model logged successfully.: 100%|██████████| 6/6 [00:26<00:00,  4.42s/it]                          
Registered CHURN_FS_MODEL V1
Alias 'production' set. Comment added.


## 12.5 Inference with Feature Retrieval

At inference time:
1. Create a spine of customer IDs to score
2. Retrieve the **latest** features from the Feature Store
3. Score using the registered production model

This guarantees the same features used in training are used for serving.

In [8]:
from snowpark_fundamentals.feature_store.training_sets import retrieve_feature_values

# Simulate: score 20 customers
inference_spine = spine_df.limit(20)

# Retrieve latest features
feature_values = retrieve_feature_values(
    fs=fs,
    spine_df=inference_spine,
    features=[rfm_fv, behavioral_fv],
)

# Load production model by alias and score
production_model = get_model_by_alias(registry, 'CHURN_FS_MODEL', 'production')
predictions = production_model.run(
    feature_values.select(FEATURE_COLS),
    function_name="predict",
)

print("Predictions for 20 customers:")
predictions.show(10)

Predictions for 20 customers:
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"DAYS_SINCE_LAST_ORDER"  |"ORDERS_30D"  |"ORDERS_90D"  |"ORDERS_365D"  |"ORDERS_TOTAL"  |"SPEND_30D"  |"SPEND_90D"  |"SPEND_365D"  |"SPEND_TOTAL"  |"AVG_ORDER_VALUE"  |"TOTAL_ITEMS"  |"AVG_ITEMS_PER_ORDER"  |"TOTAL_PAGE_VIEWS"  |"TOTAL_CLICKS"  |"TOTAL_SUPPORT_TICKETS"  |"TOTAL_EMAIL_ENGAGEMENTS"  |"INTERACTIONS_30D"  |"SUPPORT_TICKETS_30D"  |"INTERACTIONS_90D"  |"DAYS_SINCE_LAST_INTERACTION"  |"AVG_DURATION_SECONDS"  |"output_feature_0"  |
------------------------------------------------------------------------------

## 12.6 Architecture: The Complete Loop

```
┌──────────────────────────────────────────────────────────────────┐
│                        TRAINING PATH                           │
│                                                                │
│  Source Tables ──► Feature Store ──► Training Set ──► Train     │
│  (Transactions,    (RFM FV,          (spine +        (XGBoost) │
│   Interactions)     Behavioral FV)    feature join)            │
│                                                        │       │
│                                              ┌─────────▼──┐    │
│                                              │  Model     │    │
│                                              │  Registry  │    │
│                                              └─────────┬──┘    │
│                                                        │       │
│                        INFERENCE PATH                  │       │
│                                                        │       │
│  New Customer IDs ──► Feature Store ──► Retrieve ──► Score     │
│  (spine)               (same FVs)       Features   (by alias) │
│                                                        │       │
│                                                        ▼       │
│                                                   Predictions  │
└──────────────────────────────────────────────────────────────────┘
```

**Why this matters**: The same Feature Views produce features for both training
and serving. No feature skew. No manual feature engineering at inference time.

## Key Takeaways

1. The Feature Store provides **governed, reusable features** for training
2. `generate_training_set()` builds training data from registered FeatureViews
3. `retrieve_feature_values()` fetches the **same features** at inference time
4. The Model Registry stores the trained model with alias-based deployment
5. Together they guarantee **training-serving consistency** with zero feature skew

---

**Next: [13 — Production Patterns](13_production_patterns.ipynb)** — SQL inference, stored procedures, tasks, and monitoring

In [9]:
session.close()